In [ ]:
use_gpu = True
if use_gpu:
    import tensorly as tl
    tl.set_backend('pytorch')

import os
import re
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import scipy
import scanpy as sc
import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import plotnine as p9

import liana as li
from liana.mt import rank_aggregate
import cell2cell as c2c
import decoupler as dc
import omnipath as op

# Import DESeq2
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

print(sc.__version__)
print(matplotlib.__version__)
print(li.__version__)

In [ ]:
# Set this to your local analysis directory
ANALYSIS_DIR = "path/to/analysis"
os.chdir(ANALYSIS_DIR)

# load
full_dataset = sc.read_h5ad("seurat_objects/full_dataset_for_LIANA_processed.h5ad")
full_dataset.obs["response_timepoint"] = full_dataset.obs["response"].astype(str) + "_" + full_dataset.obs["timepoint2"].astype(str)
full_dataset.obs["sample_timepoint2"] = full_dataset.obs["patient.id"].astype(str) + "_" + full_dataset.obs["timepoint2"].astype(str)

F_FULL_RES = "FullDataset_LIANA_bysample_TDN_D0.csv"
liana_path = 'python_data/LIANA_CSVs/' + F_FULL_RES 

# 1. Add the saved results back into the object
full_dataset.uns['liana_res'] = pd.read_csv(liana_path)
res = full_dataset.uns['liana_res'].copy()

In [ ]:
# --- helpers ---------------------------------------------------------------
def subunits(col):
    """Split complex strings (e.g. 'CD8A_CD8B') into sets of genes."""
    return col.astype(str).str.split('_').apply(set)

def all_in(col, ref):
    """True if every subunit of the complex is in ref."""
    return subunits(col).apply(lambda g: g <= ref)

def has_gene(col, genes):
    """True if any subunit of the complex is in genes."""
    return subunits(col).apply(lambda g: bool(g & genes))

# 2. Keep cell-surface receptors and surface/secreted ligands
#    (all subunits of a complex must pass)
surface_receptors = set(pd.read_csv('python_data/consensus_cell_surface_receptors_2.csv')['receptor'])
surface_ligands   = set(pd.read_csv('python_data/consensus_extracellular_ligands.csv')['ligand'])

res = res[
    res["ligand_complex"].isin(surface_ligands) &
    res["receptor_complex"].isin(surface_receptors)
]

# 3. Lineage-consistent co-receptor usage (receptor is expressed by the target cell)
#    Labels here must match the cell type labels in the LIANA input
#    (they are renamed for display further below via rename_map).
cd4_ok = {'CD14 Mono', 'CD16 Mono', 'Dendritic Cell', 'Treg_CAR', 'NK_CAR', 'CD4 Naive','CD4 Naive_CAR',
          'CD4 TEM-like', 'CD4 TEM-like_CAR', 'Treg', 'Proliferating', 'Proliferating_CAR', "MAIT"}
cd8_ok = {'CD8 TEM', 'CD8 Naive', 'gdT', 'NK', 'Proliferating', 'CD8 TEM_CAR', 
          'CD8 Naive_CAR', 'gdT_CAR', 'Proliferating_CAR', "MAIT"}

def lineage_mismatch(df):
    tgt = df['target'].astype(str)
    cd8_rec = has_gene(df['receptor_complex'], {'CD8A', 'CD8B'})
    cd4_rec = has_gene(df['receptor_complex'], {'CD4'})
    return (cd8_rec & ~tgt.isin(cd8_ok)) | (cd4_rec & ~tgt.isin(cd4_ok))

res = res[~lineage_mismatch(res)].reset_index(drop=True)

# 4. Store and save the filtered version (the unfiltered one stays in uns['liana_res'])
full_dataset.uns['liana_res_surface'] = res
res.to_csv(liana_path.replace('.csv', '_surface_noCD4CD8mismatch.csv'), index=False)

# Check
unmatched = set(full_dataset.uns['liana_res']['target'].astype(str)) - (cd4_ok | cd8_ok)
print(f"{len(full_dataset.uns['liana_res'])} -> {len(res)} rows | "
      f"non-surface receptors left: {(~all_in(res['receptor_complex'], surface_receptors)).sum()} | "
      f"non-extracellular ligands left: {(~all_in(res['ligand_complex'], surface_ligands)).sum()} | "
      f"CD4/CD8 mismatches left: {lineage_mismatch(res).sum()}")
print(f"target labels not in either allow-list: {unmatched or 'none'}")

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
OBJECTS_DIR = os.path.join(ANALYSIS_DIR, "python_objects")
FIG_DIR = os.path.join(ANALYSIS_DIR, "python_data", "images", "Tensor",
                       "FullDataset_TDN_D0_bysample_strict_receptor_ligand")
 
for d in (OBJECTS_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)
    
SAMPLE_KEY = "sample_timepoint2"
CONTEXT_KEY = "response_timepoint"
SCORE_KEY = "magnitude_rank"
 
F_TENSOR = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_strict_receptor_ligand.pkl")
F_TENSOR_META = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_metadata_strict_receptor_ligand.pkl")
F_TENSOR_FACTORIZED = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_factorized_strict_receptor_ligand.pkl")

sorted_samples = sorted(full_dataset.uns['liana_res']['sample_timepoint2'].unique())

full_tensor_all = li.multi.to_tensor_c2c(liana_res=full_dataset.uns['liana_res_surface'], # LIANA's dataframe containing results
                                sample_key='sample_timepoint2', # Column name of the samples
                                source_key='source', # Column name of the sender cells
                                target_key='target', # Column name of the receiver cells
                                ligand_key='ligand_complex', # Column name of the ligands
                                receptor_key='receptor_complex', # Column name of the receptors
                                score_key='magnitude_rank', # Column name of the communication scores to use
                                inverse_fun=lambda x: 1 - x, # Transformation function
                                how='outer', # What to include across all samples
                                outer_fraction=0.15, # Fraction of samples as threshold to include cells and LR pairs.
                                context_order=sorted_samples, # Order to store the contexts in the tensor
                                #lr_fill=np.nan, # What to fill missing LRs with
                                #cell_fill = np.nan, # What to fill missing cell types with
                                lr_sep='-', # How to separate ligand and receptor names to name LR pair
                                #sort_elements=True # Whether sorting alphabetically element names of each tensor dim. Does not apply for context order if context_order is passed.
                              )

In [ ]:
# ---- Tensor metadata --------------------------------------------------------
full_dataset.obs[CONTEXT_KEY] = (full_dataset.obs["response"].astype(str) + "_"
                                 + full_dataset.obs["timepoint2"].astype(str))
 
context_dict = full_dataset.obs.sort_values(by=SAMPLE_KEY) \
                        .set_index(SAMPLE_KEY)[CONTEXT_KEY] \
                        .to_dict()
 
dimensions_dict = [context_dict, None, None, None]
meta_tensor_full = c2c.tensor.generate_tensor_metadata(interaction_tensor=full_tensor_all,
                                                  metadata_dicts=dimensions_dict,
                                                  fill_with_order_elements=True
                                                 )
 
# save
c2c.io.export_variable_with_pickle(variable=full_tensor_all, filename=F_TENSOR)
c2c.io.export_variable_with_pickle(variable=meta_tensor_full, filename=F_TENSOR_META)

In [ ]:
# ---- Tensor factorization ---------------------------------------------------
full_tensor_factorized = c2c.analysis.run_tensor_cell2cell_pipeline(full_tensor_all,
                                           meta_tensor_full,
                                           copy_tensor=True,
                                           rank=None, # Number of factors to perform the factorization. If None, it is automatically determined by an elbow analysis
                                           tf_optimization='robust', # To define how robust we want the analysis to be.
                                           random_state=0, # Random seed for reproducibility
                                           upper_rank=25,
                                           device='cuda', # Device to use. If using GPU and PyTorch, use 'cuda'. For CPU use 'cpu'
                                           output_folder=FIG_DIR, # Whether to save the figures in files. If so, a folder pathname must be passed
                                          )
 
c2c.io.export_variable_with_pickle(variable=full_tensor_factorized, filename=F_TENSOR_FACTORIZED)
 
# ---- Resume from disk (only needed in a fresh session) ----------------------
# full_tensor_all = c2c.io.load_variable_with_pickle(filename=F_TENSOR)
# meta_tensor_full = c2c.io.load_variable_with_pickle(filename=F_TENSOR_META)
 
print(full_tensor_all.shape)
print(full_tensor_all.missing_fraction())
print(full_tensor_all.sparsity_fraction())
print(full_tensor_all.excluded_value_fraction())

In [ ]:
# ---- CONFIG -----------------------------------------------------------------
OBJECTS_DIR = os.path.join(ANALYSIS_DIR, "python_objects")
FIG_DIR = os.path.join(ANALYSIS_DIR, "python_data", "images", "Tensor",
                       "FullDataset_TDN_D0_bysample_strict_receptor_ligand")
TENSOR_DIR = os.path.join(ANALYSIS_DIR, "python_data", "images", "Tensor",
                          "FullDataset_TDN_D0_bysample_strict_receptor_ligand")
 
os.makedirs(TENSOR_DIR, exist_ok=True)
 
SAMPLE_KEY = "sample_timepoint2"
CONTEXT_KEY = "response_timepoint"
 
F_TENSOR_FACTORIZED = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_factorized_strict_receptor_ligand.pkl")
F_TENSOR_META = os.path.join(OBJECTS_DIR, "fulldataset_tensor_bysample_TDN_D0_metadata_strict_receptor_ligand.pkl")
 
# response_timepoint levels, in plotting order (CR = complete response, PD = progressive disease)
CONTEXT_ORDER = ["CR_APH", "PD_APH",
                 "CR_TDN-D0", "PD_TDN-D0",
                 "CR_Peak", "PD_Peak",
                 "CR_4W", "PD_4W"]
 
TOP_N = 20
LOADING_THRESHOLD = 0.1

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

In [ ]:
# ---- Rebuild the grouping columns -------------------------------------------
full_dataset.obs["patient.id"] = full_dataset.obs["patient.id"].astype(str)
full_dataset.obs["timepoint2"] = full_dataset.obs["timepoint2"].astype(str)
full_dataset.obs["response"] = full_dataset.obs["response"].astype(str)
 
full_dataset.obs[SAMPLE_KEY] = full_dataset.obs["patient.id"] + "_" + full_dataset.obs["timepoint2"]
full_dataset.obs[CONTEXT_KEY] = full_dataset.obs["response"] + "_" + full_dataset.obs["timepoint2"]
print(Counter(full_dataset.obs[SAMPLE_KEY]))
print(Counter(full_dataset.obs[CONTEXT_KEY]))
context_dict = full_dataset.obs.sort_values(by=SAMPLE_KEY) \
                       .set_index(SAMPLE_KEY)[CONTEXT_KEY] \
                       .to_dict()
dimensions_dict = [context_dict, None, None, None]

factors = c2c.io.load_tensor_factors(os.path.join(TENSOR_DIR, "Loadings.xlsx"))

In [ ]:
 # ----  (REDONE IN R) Context boxplots with Mann-Whitney tests -------------------------------
_ = c2c.plotting.context_boxplot(context_loadings=full_tensor_factorized.factors['Contexts'],
                                 metadict=context_dict,
                                 nrows=2,
                                 figsize=(10, 20),
                                 statistical_test='Mann-Whitney',
                                 pval_correction=None,
                                 text_format="simple",
                                 cmap='plasma',
                                 verbose=False,
                                 # group_order=CONTEXT_ORDER,
                                 filename=os.path.join(TENSOR_DIR, "Factor_Boxplots_Wilcoxon.pdf")
                                )

# Input label -> display name (only list the ones you want to change)
rename_map = {
    "CD4 TEM-like_CAR": "CAR+ CD4 TEM-like"
}

factors_plot = {k: v.copy() for k, v in factors.items()}
for dim in ["Sender Cells", "Receiver Cells"]:
    factors_plot[dim] = factors_plot[dim].rename(index=rename_map)
    assert factors_plot[dim].index.is_unique, f"duplicate names in {dim} after renaming"

pd.Series(rename_map, name="display_name").to_csv(os.path.join(TENSOR_DIR, "celltype_rename_map.csv"))

In [ ]:
# ---- Top ligand-receptor pairs per factor -----------------------------------
lr_loadings = factors_plot['Ligand-Receptor Pairs']
#lr_loadings.to_csv(os.path.join(TENSOR_DIR, "LR_loading_values.csv"))
 

for fn in range(1, lr_loadings.shape[1] + 1):
    factor = "Factor " + str(fn)
    print(lr_loadings.sort_values(factor, ascending=False).head(TOP_N))
    
## all factors: keep the top LRs of each
rows_to_keep = []
for col in lr_loadings.columns:
    top_rows = lr_loadings[lr_loadings[col] > LOADING_THRESHOLD].nlargest(TOP_N, col)
    rows_to_keep.extend(top_rows.index.tolist())
    
## remove duplicates while preserving order
rows_to_keep = list(dict.fromkeys(rows_to_keep))
df_filtered = lr_loadings.loc[rows_to_keep]
print(df_filtered.shape)
 
lr_cm = c2c.plotting.loading_clustermap(df_filtered,
                                        loading_threshold=LOADING_THRESHOLD, # To consider only top LRs
                                        figsize=(28, 8),
                                        filename=os.path.join(TENSOR_DIR, "Clustermap-LRs_filtered.pdf"),
                                        row_cluster=False # To avoid clustering of factors
                                       )

# Editable text in Illustrator - keep this in the same cell as the plots
matplotlib.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
})

def save_fig(fig, name):
    """Save a figure as PDF + SVG with editable text, then close it."""
    for ext in ("pdf", "svg"):
        fig.savefig(os.path.join(TENSOR_DIR, f"{name}.{ext}"), bbox_inches="tight")
    plt.close(fig)

In [ ]:
# ---- Sender x receiver joint loadings, per factor ---------------------------
for selected_factor in factors_plot["Contexts"].columns:
    print(selected_factor)
    loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(factors_plot,
                                                                        dim1='Sender Cells',
                                                                        dim2='Receiver Cells',
                                                                        factor=selected_factor
                                                                       )

    lprod_cm = c2c.plotting.loading_clustermap(loading_product.T, # Remove .T to transpose the axes
                                               use_zscore=False,  # Whether standardizing the loadings across factors
                                               figsize=(8, 8),
                                               cbar_label='Loading Product',
                                              )                   # no filename: saved below
    save_fig(lprod_cm.fig, selected_factor.replace(" ", "_") + "_Clustermap-CC-Pairs")

In [ ]:
# ---- Factor-specific communication networks ---------------------------------
# Get all outer products as adjacency matrices, one per factor
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(factors_plot,
                                                                           sender_label='Sender Cells',
                                                                           receiver_label='Receiver Cells',
                                                                          )
# Then, flatten the adjacency matrices
network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby='receivers')

# Distribution of the weights for each factor-specific network
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(network_by_factors.values.flatten(), bins=50, color='skyblue', edgecolor='black')
ax.set_title("Distribution of factor-specific network weights")
ax.set_xlabel("Edge weight")
ax.set_ylabel("Frequency")
fig.tight_layout()
save_fig(fig, "Factor-Network_weight_distribution")
 

matplotlib.rcParams.update({
    "pdf.fonttype": 42,          # embed TrueType -> editable text in PDF
    "ps.fonttype": 42,
    "svg.fonttype": "none",      # keep text as <text> in SVG
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
})

c2c.plotting.ccc_networks_plot(factors_plot,
                               included_factors=factors_plot["Contexts"].columns,
                               ccc_threshold=0.05,
                               nrows=1,
                               panel_size=(16, 16))   # no filename: save it ourselves

fig = plt.gcf()
fig.savefig(os.path.join(TENSOR_DIR, "Factor-Networks.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(TENSOR_DIR, "Factor-Networks.svg"), bbox_inches="tight")

# Editable text in Illustrator - keep this in the same cell as the plots
matplotlib.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
})

# selected factors only for more focused results
cols = ["Factor 5", "Factor 6"]

rows_to_keep = []
for col in cols:
    top_rows = lr_loadings[lr_loadings[col] > LOADING_THRESHOLD].nlargest(TOP_N, col)
    rows_to_keep.extend(top_rows.index.tolist())

rows_to_keep = list(dict.fromkeys(rows_to_keep))
df_filtered_2 = lr_loadings.loc[rows_to_keep]
print(df_filtered_2.shape)

lr_cm = c2c.plotting.loading_clustermap(df_filtered_2,
                                        loading_threshold=LOADING_THRESHOLD,
                                        use_zscore=False,
                                        figsize=(28, 8),
                                        row_cluster=False
                                       )
save_fig(lr_cm.fig, "Clustermap-LRs_top20_factor5-6-only")

# unfiltered
lr_cm = c2c.plotting.loading_clustermap(lr_loadings,
                                        loading_threshold=LOADING_THRESHOLD,
                                        use_zscore=False,
                                        figsize=(28, 8),
                                        row_cluster=False
                                       )
save_fig(lr_cm.fig, "Clustermap-LRs")

In [ ]:
# ---- Gini coefficients ------------------------------------------------------
gini_df = c2c.analysis.tensor_downstream.compute_gini_coefficients(factors,
                                                         sender_label='Sender Cells',
                                                         receiver_label='Receiver Cells'
                                                        )
 
gini_df.to_csv(os.path.join(TENSOR_DIR, "gini_df.csv"))
print(gini_df)